### Config and setup

#### Function refresh

In [ ]:
%load_ext autoreload
%autoreload 2
# %reload_ext autoreload

In [ ]:
import sys
sys.path.insert(0, '../functions')

#### Library imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
import json
import random
from pathlib import Path
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from scipy import stats
from scipy.stats import mannwhitneyu
from scipy import stats

import time

import seaborn as sns
import matplotlib.pyplot as plt

from captum.attr import IntegratedGradients

torch.manual_seed(25)
np.random.seed(25)
random.seed(25)

#### Functions import

In [ ]:
from models import DANN
from data_loading import prepare_dataloaders, load_subject_data_flexible
from metrics import evaluate

from explainability import (
    DANNClassifierWrapper, load_model_for_xai,
    fold_batchnorm, verify_folding,
    lrp_epsilon_conv2d, lrp_zplus_conv2d,
    lrp_linear, lrp_avgpool,
    compute_lrp, run_lrp_full,
    load_lrp_attributions,
    compute_per_class_accuracy_106,
    compute_deeplift, run_deeplift_full,
    load_deeplift_attributions,
    compute_shap, run_shap_full,
    load_shap_attributions,
    compute_ig, run_ig_full,
    load_ig_attributions,
    run_misclassified_single,
    run_and_save_misclassified
)

from xai_utils import (
    assign_score_groups, get_grouped_order,
    get_score_grouped_order, classify_failure_groups,
    compute_lrp_deeplift_agreement,
    compute_method_agreement,
    compute_within_group_consistency,
    print_consistency_results,
    ch16_list,
    build_binary_relevance_table,
    build_multiclass_relevance_table,
    get_top_bottom_channels
)

from xai_plots import (
    plot_group_mean_attribution,
    plot_stopping_criterion_consistency
)

#### Path

In [ ]:
base_path = Path("..")
processed_path = base_path / "Data/processed"
results_path = base_path / "results"
ch16_path = base_path / "Data/processed_16ch"

# existing attribution folders for loading LRP and DeepLIFT
lrp_attr_path = results_path / "lrp_attributions"
binary_attr_path = lrp_attr_path / "binary_cdan_16ch"
multiclass_attr_path = lrp_attr_path / "multiclass_16ch"

deeplift_attr_path = results_path / "deeplift_attributions"
binary_deeplift_path = deeplift_attr_path / "binary_cdan_16ch"
multiclass_deeplift_path = deeplift_attr_path / "multiclass_16ch"

per_class_path = results_path / "per_class_accuracy"

shap_attr_path = results_path / "shap_attributions"
binary_shap_path = shap_attr_path / "binary_cdan_16ch"
multiclass_shap_path = shap_attr_path / "multiclass_16ch"

# new IG folders
ig_attr_path = results_path / "ig_attributions"
binary_ig_zero_path = ig_attr_path / "binary_cdan_16ch_zero"
binary_ig_mean_path = ig_attr_path / "binary_cdan_16ch_mean"
multiclass_ig_zero_path = ig_attr_path / "multiclass_16ch_zero"
multiclass_ig_mean_path = ig_attr_path / "multiclass_16ch_mean"

# only creating zero baseline folders since mean is dropped
binary_ig_zero_path.mkdir(parents=True, exist_ok=True)
multiclass_ig_zero_path.mkdir(parents=True, exist_ok=True)

# misclassified data path
binary_misc_path = results_path / "misclassified_attributions" / "binary_cdan_16ch"
multiclass_misc_path = results_path / "misclassified_attributions" / "multiclass_16ch"

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

#### 16 channel importing from xai_utils

In [ ]:
ch16_list = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4',
             'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6']

#### Valid Subjects

In [ ]:
# 106 subjects after removing S088, S092, S100 (incompatible sampling rate)
excluded = ['S088', 'S092', 'S100']

valid_subjects = []
for i in range(1, 110):
    subject_id = f'S{str(i).zfill(3)}'
    subject_file = processed_path / f'{subject_id}_processed.npz'
    if subject_file.exists() and subject_id not in excluded:
        valid_subjects.append(subject_id)

print(f"Valid subjects: {len(valid_subjects)}")
print(f"First five: {valid_subjects[:5]}")
print(f"Last five: {valid_subjects[-5:]}")

#### Setup cell for groupings rebuild

In [ ]:
consistency_subjects = ['S007', 'S085', 'S108', 'S107', 'S080',
                         'S097', 'S037', 'S006', 'S039', 'S018']

In [ ]:
# load kappas
with open(results_path / 'loso_results_cdan_16ch_RETRAIN.json') as f:
    cdan_results = json.load(f)

all_kappas = {
    subj: cdan_results[subj]['kappa_stopping']['final_kappa']
    for subj in valid_subjects
}

# load per-class accuracy
with open(per_class_path / 'multiclass_16ch_per_class_accuracy.json') as f:
    per_class_data = json.load(f)

all_per_class = {}
for subj in valid_subjects:
    pca = per_class_data[subj]['kappa']['per_class_accuracy']
    all_per_class[subj] = {int(cls): float(acc) for cls, acc in pca.items()}

# rebuild groupings
score_groups, boundaries = assign_score_groups(all_kappas, std_multiplier=0.7)

all_kappas_binary = {subj: cdan_results[subj]['kappa_stopping']['final_kappa']
                     for subj in valid_subjects}
group_a, group_b, neither = classify_failure_groups(all_per_class, all_kappas_binary)
group_a_subjects = [s['subject'] for s in group_a]
group_b_subjects = [s['subject'] for s in group_b]

category_groups_106 = {}
for subj in valid_subjects:
    if all_kappas[subj] < 0.1:
        category_groups_106[subj] = 'negative_transfer'
    elif subj in group_a_subjects:
        category_groups_106[subj] = 'group_a_collapse'
    elif subj in group_b_subjects:
        category_groups_106[subj] = 'group_b_flat'
    else:
        category_groups_106[subj] = 'neither'

xai_subjects = {
    "top": ["S007", "S085"],
    "bottom_kappa": ["S018", "S097", "S107"],
    "group_a_collapse": ["S037", "S006"],
    "group_b_flat": ["S039", "S020"],
    "typical": ["S080", "S065", "S030", "S108"],
}
xai_subset = [s for group in xai_subjects.values() for s in group]

# load existing attributions for comparison
binary_attrs_106 = load_lrp_attributions(
    binary_attr_path, valid_subjects, n_classes=2)

mc_attrs_106 = load_lrp_attributions(
    multiclass_attr_path, valid_subjects, n_classes=4)

binary_dl_106 = load_deeplift_attributions(
    binary_deeplift_path, valid_subjects, n_classes=2)

mc_dl_106 = load_deeplift_attributions(
    multiclass_deeplift_path, valid_subjects, n_classes=4)

binary_shap_106 = load_shap_attributions(
    binary_shap_path, valid_subjects, n_classes=2)

mc_shap_106 = load_shap_attributions(
    multiclass_shap_path, valid_subjects, n_classes=4)

binary_ig_106 = load_ig_attributions(
    binary_ig_zero_path, valid_subjects, n_classes=2)

mc_ig_106 = load_ig_attributions(
    multiclass_ig_zero_path, valid_subjects, n_classes=4)

print("Setup complete")
print(f"Score groups: {dict(Counter(score_groups.values()))}")
print(f"Category groups: {dict(Counter(category_groups_106.values()))}")
print(f"XAI subset: {len(xai_subset)} subjects")
print(f"LRP binary: {len(binary_attrs_106)}, DeepLIFT: {len(binary_dl_106)}, "
      f"SHAP: {len(binary_shap_106)}, IG: {len(binary_ig_106)}")

### End of config and setup

### 1. Group-level mean attribution: Binary (all four methods)

#### # LRP: binary

In [ ]:
# LRP: binary
plot_group_mean_attribution(
    binary_attrs_106, category_groups_106,
    score_groups, valid_subjects, all_kappas,
    title_tag='LRP binary CDAN-16ch'
)

#### # DeepLIFT: binary, mean baseline

In [ ]:
# DeepLIFT: binary, mean baseline
plot_group_mean_attribution(
    binary_dl_106, category_groups_106,
    score_groups, valid_subjects, all_kappas,
    title_tag='DeepLIFT binary CDAN-16ch'
)

#### # SHAP: binary, 60% source pool, seed=25

In [ ]:
# SHAP: binary, 60% source pool, seed=25
plot_group_mean_attribution(
    binary_shap_106, category_groups_106,
    score_groups, valid_subjects, all_kappas,
    title_tag='GradientSHAP binary CDAN-16ch'
)

#### # IG: binary, zero baseline, n_steps=50

In [ ]:
# IG: binary, zero baseline, n_steps=50
plot_group_mean_attribution(
    binary_ig_106, category_groups_106,
    score_groups, valid_subjects, all_kappas,
    title_tag='IG zero baseline binary CDAN-16ch'
)

### 2. Group-level mean attribution: Multiclass (all four methods)  

#### # LRP: multiclass

In [ ]:
# LRP: multiclass
plot_group_mean_attribution(
    mc_attrs_106, category_groups_106,
    score_groups, valid_subjects, all_kappas,
    title_tag='LRP multiclass-16ch'
)

#### # DeepLIFT: multiclass, mean baseline

In [ ]:
# DeepLIFT: multiclass, mean baseline
plot_group_mean_attribution(
    mc_dl_106, category_groups_106,
    score_groups, valid_subjects, all_kappas,
    title_tag='DeepLIFT multiclass-16ch'
)

#### # SHAP: multiclass, 60% source pool, seed=25

In [ ]:
# SHAP: multiclass, 60% source pool, seed=25
plot_group_mean_attribution(
    mc_shap_106, category_groups_106,
    score_groups, valid_subjects, all_kappas,
    title_tag='GradientSHAP multiclass-16ch'
)

In [ ]:
# IG — multiclass, zero baseline, n_steps=50
plot_group_mean_attribution(
    mc_ig_106, category_groups_106,
    score_groups, valid_subjects, all_kappas,
    title_tag='IG zero baseline multiclass-16ch'
)

### 3. Four-method agreement analysis

#### # LRP vs DeepLIFT, LRP vs Shap, LRP vs IG

In [ ]:
# LRP vs DeepLIFT
print("LRP vs DeepLIFT agreement, binary, 106 subjects")
print("-" * 50)
agreement_lrp_dl = compute_method_agreement(
    binary_attrs_106, binary_dl_106,
    valid_subjects, criterion='kappa', n_classes=2
)
for cls, name in enumerate(['left', 'right']):
    correlations = [agreement_lrp_dl[s][cls] for s in valid_subjects
                    if agreement_lrp_dl.get(s, {}).get(cls) is not None]
    print(f"Class {name}: mean_r={np.mean(correlations):.4f}, "
          f"std={np.std(correlations):.4f}, "
          f"min={np.min(correlations):.4f}, "
          f"max={np.max(correlations):.4f}")

# LRP vs SHAP
print()
print("LRP vs SHAP agreement, binary, 106 subjects")
print("-" * 50)
agreement_lrp_shap = compute_method_agreement(
    binary_attrs_106, binary_shap_106,
    valid_subjects, criterion='kappa', n_classes=2
)
for cls, name in enumerate(['left', 'right']):
    correlations = [agreement_lrp_shap[s][cls] for s in valid_subjects
                    if agreement_lrp_shap.get(s, {}).get(cls) is not None]
    print(f"Class {name}: mean_r={np.mean(correlations):.4f}, "
          f"std={np.std(correlations):.4f}, "
          f"min={np.min(correlations):.4f}, "
          f"max={np.max(correlations):.4f}")

# LRP vs IG
print()
print("LRP vs IG agreement, binary, 106 subjects")
print("-" * 50)
agreement_lrp_ig = compute_method_agreement(
    binary_attrs_106, binary_ig_106,
    valid_subjects, criterion='kappa', n_classes=2
)
for cls, name in enumerate(['left', 'right']):
    correlations = [agreement_lrp_ig[s][cls] for s in valid_subjects
                    if agreement_lrp_ig.get(s, {}).get(cls) is not None]
    print(f"Class {name}: mean_r={np.mean(correlations):.4f}, "
          f"std={np.std(correlations):.4f}, "
          f"min={np.min(correlations):.4f}, "
          f"max={np.max(correlations):.4f}")

#### # DeepLIFT vs SHAP and DeepLIFT vs IG

In [ ]:
# DeepLIFT vs SHAP and DeepLIFT vs IG
print("DeepLIFT vs SHAP agreement, binary, 106 subjects")
print("-" * 50)
agreement_dl_shap = compute_method_agreement(
    binary_dl_106, binary_shap_106,
    valid_subjects, criterion='kappa', n_classes=2
)
for cls, name in enumerate(['left', 'right']):
    correlations = [agreement_dl_shap[s][cls] for s in valid_subjects
                    if agreement_dl_shap.get(s, {}).get(cls) is not None]
    print(f"Class {name}: mean_r={np.mean(correlations):.4f}, "
          f"std={np.std(correlations):.4f}, "
          f"min={np.min(correlations):.4f}, "
          f"max={np.max(correlations):.4f}")

print()
print("DeepLIFT vs IG agreement, binary, 106 subjects")
print("-" * 50)
agreement_dl_ig = compute_method_agreement(
    binary_dl_106, binary_ig_106,
    valid_subjects, criterion='kappa', n_classes=2
)
for cls, name in enumerate(['left', 'right']):
    correlations = [agreement_dl_ig[s][cls] for s in valid_subjects
                    if agreement_dl_ig.get(s, {}).get(cls) is not None]
    print(f"Class {name}: mean_r={np.mean(correlations):.4f}, "
          f"std={np.std(correlations):.4f}, "
          f"min={np.min(correlations):.4f}, "
          f"max={np.max(correlations):.4f}")

#### with multiclass

In [ ]:
# Remaining binary pairs
print("DeepLIFT vs IG agreement, binary, 106 subjects")
print("-" * 50)
agreement_dl_ig = compute_method_agreement(
    binary_dl_106, binary_ig_106,
    valid_subjects, criterion='kappa', n_classes=2
)
for cls, name in enumerate(['left', 'right']):
    correlations = [agreement_dl_ig[s][cls] for s in valid_subjects
                    if agreement_dl_ig.get(s, {}).get(cls) is not None]
    print(f"Class {name}: mean_r={np.mean(correlations):.4f}, "
          f"std={np.std(correlations):.4f}")

print()
print("DeepLIFT vs SHAP agreement, binary, 106 subjects")
print("-" * 50)
agreement_dl_shap = compute_method_agreement(
    binary_dl_106, binary_shap_106,
    valid_subjects, criterion='kappa', n_classes=2
)
for cls, name in enumerate(['left', 'right']):
    correlations = [agreement_dl_shap[s][cls] for s in valid_subjects
                    if agreement_dl_shap.get(s, {}).get(cls) is not None]
    print(f"Class {name}: mean_r={np.mean(correlations):.4f}, "
          f"std={np.std(correlations):.4f}")

print()
print("SHAP vs IG agreement, binary, 106 subjects")
print("-" * 50)
agreement_shap_ig = compute_method_agreement(
    binary_shap_106, binary_ig_106,
    valid_subjects, criterion='kappa', n_classes=2
)
for cls, name in enumerate(['left', 'right']):
    correlations = [agreement_shap_ig[s][cls] for s in valid_subjects
                    if agreement_shap_ig.get(s, {}).get(cls) is not None]
    print(f"Class {name}: mean_r={np.mean(correlations):.4f}, "
          f"std={np.std(correlations):.4f}")

print()
print("=" * 50)
print("MULTICLASS — all six pairs, all four classes")
print("=" * 50)

mc_method_pairs = [
    ("LRP vs DeepLIFT", mc_attrs_106, mc_dl_106),
    ("LRP vs IG", mc_attrs_106, mc_ig_106),
    ("LRP vs SHAP", mc_attrs_106, mc_shap_106),
    ("DeepLIFT vs IG", mc_dl_106, mc_ig_106),
    ("DeepLIFT vs SHAP", mc_dl_106, mc_shap_106),
    ("SHAP vs IG", mc_shap_106, mc_ig_106)
]

class_names = ['left', 'right', 'both_fists', 'both_feet']

for pair_name, dict1, dict2 in mc_method_pairs:
    print(f"\n{pair_name}")
    print("-" * 50)
    agreement = compute_method_agreement(
        dict1, dict2,
        valid_subjects, criterion='kappa', n_classes=4
    )
    for cls, name in enumerate(class_names):
        correlations = [agreement[s][cls] for s in valid_subjects
                       if agreement.get(s, {}).get(cls) is not None]
        print(f"Class {name}: mean_r={np.mean(correlations):.4f}, "
              f"std={np.std(correlations):.4f}")

### 4. Within-group consistency (LRP, DeepLIFT, SHAP)
Within-group consistency is a measure of how similar two subjects' attribution patterns are to each other. The higher the number, the more subjects in that group agree on which channels and time points matter.

#### Binary

In [ ]:
# LRP within-group consistency
print("Within-group consistency | LRP binary | category grouping")
print("-" * 50)
res_lrp_wg = compute_within_group_consistency(
    binary_attrs_106, category_groups_106,
    valid_subjects, criterion='kappa', n_classes=2
)
print_consistency_results(res_lrp_wg, n_classes=2,
                           class_names=['left', 'right'])

In [ ]:
# DeepLIFT within-group consistency
print("Within-group consistency | DeepLIFT binary | category grouping")
print("-" * 50)
res_dl_wg = compute_within_group_consistency(
    binary_dl_106, category_groups_106,
    valid_subjects, criterion='kappa', n_classes=2
)
print_consistency_results(res_dl_wg, n_classes=2,
                           class_names=['left', 'right'])

In [ ]:
# SHAP within-group consistency
print("Within-group consistency | SHAP binary | category grouping")
print("-" * 50)
res_shap_wg = compute_within_group_consistency(
    binary_shap_106, category_groups_106,
    valid_subjects, criterion='kappa', n_classes=2
)
print_consistency_results(res_shap_wg, n_classes=2,
                           class_names=['left', 'right'])

In [ ]:
# IG within-group consistency
print("Within-group consistency | IG binary | category grouping")
print("-" * 50)
res_ig_wg = compute_within_group_consistency(
    binary_ig_106, category_groups_106,
    valid_subjects, criterion='kappa', n_classes=2
)
print_consistency_results(res_ig_wg, n_classes=2,
                           class_names=['left', 'right'])

#### Multiclass

In [ ]:
class_names_mc = ['left', 'right', 'both_fists', 'both_feet']

# LRP multiclass
print("Within-group consistency | LRP multiclass | category grouping")
print("-" * 50)
res_lrp_mc_wg = compute_within_group_consistency(
    mc_attrs_106, category_groups_106,
    valid_subjects, criterion='kappa', n_classes=4
)
print_consistency_results(res_lrp_mc_wg, n_classes=4,
                          class_names=class_names_mc)

In [ ]:
# DeepLIFT multiclass
print("Within-group consistency | DeepLIFT multiclass | category grouping")
print("-" * 50)
res_dl_mc_wg = compute_within_group_consistency(
    mc_dl_106, category_groups_106,
    valid_subjects, criterion='kappa', n_classes=4
)
print_consistency_results(res_dl_mc_wg, n_classes=4,
                          class_names=class_names_mc)

In [ ]:
# SHAP multiclass
print("Within-group consistency | SHAP multiclass | category grouping")
print("-" * 50)
res_shap_mc_wg = compute_within_group_consistency(
    mc_shap_106, category_groups_106,
    valid_subjects, criterion='kappa', n_classes=4
)
print_consistency_results(res_shap_mc_wg, n_classes=4,
                          class_names=class_names_mc)

In [ ]:
# IG multiclass
print("Within-group consistency | IG multiclass | category grouping")
print("-" * 50)
res_ig_mc_wg = compute_within_group_consistency(
    mc_ig_106, category_groups_106,
    valid_subjects, criterion='kappa', n_classes=4
)
print_consistency_results(res_ig_mc_wg, n_classes=4,
                          class_names=class_names_mc)

### 5. Stopping criterion consistency (all four methods)

In [ ]:
# LRP stopping criterion consistency
plot_stopping_criterion_consistency(
    binary_attrs_106, binary_attrs_106,
    subjects=consistency_subjects,
    n_classes=2, title_tag='LRP binary'
)

In [ ]:
# DeepLIFT stopping criterion consistency
plot_stopping_criterion_consistency(
    binary_dl_106, binary_dl_106,
    subjects=consistency_subjects,
    n_classes=2, title_tag='DeepLIFT binary'
)

In [ ]:
# SHAP stopping criterion consistency
plot_stopping_criterion_consistency(
    binary_shap_106, binary_shap_106,
    subjects=consistency_subjects,
    n_classes=2, title_tag='GradientSHAP binary'
)

In [ ]:
# IG stopping criterion consistency
plot_stopping_criterion_consistency(
    binary_ig_106, binary_ig_106,
    subjects=consistency_subjects,
    n_classes=2, title_tag='IG binary'
)

### 6. Commented-out generation code (reference only)

GENERATION CODE

All attribution files already saved to disk.

Uncomment and run only if files need to be regenerated from scratch.

In [ ]:
# --- LRP ---
# binary_completed, _, _ = run_lrp_full(
#     valid_subjects=valid_subjects,
#     checkpoint_tag='cdan_16ch_binary',
#     attr_save_path=binary_attr_path,
#     n_classes=2, use_multiclass=False,
# )
# mc_completed, _, _ = run_lrp_full(
#     valid_subjects=valid_subjects,
#     checkpoint_tag='multiclass_16ch',
#     attr_save_path=multiclass_attr_path,
#     n_classes=4, use_multiclass=True,
# )

In [ ]:
# --- DeepLIFT ---
# run_deeplift_full(
#     valid_subjects=valid_subjects,
#     checkpoint_tag='cdan_16ch_binary',
#     attr_save_path=binary_deeplift_path,
#     n_classes=2, use_multiclass=False, baseline='mean',
# )
# run_deeplift_full(
#     valid_subjects=valid_subjects,
#     checkpoint_tag='multiclass_16ch',
#     attr_save_path=multiclass_deeplift_path,
#     n_classes=4, use_multiclass=True, baseline='mean',
# )

In [ ]:
# --- SHAP ---
# run_shap_full(
#     valid_subjects=valid_subjects,
#     checkpoint_tag='cdan_16ch_binary',
#     attr_save_path=binary_shap_path,
#     n_classes=2, use_multiclass=False,
#     background_fraction=0.60, seed=25,
# )
# run_shap_full(
#     valid_subjects=valid_subjects,
#     checkpoint_tag='multiclass_16ch',
#     attr_save_path=multiclass_shap_path,
#     n_classes=4, use_multiclass=True,
#     background_fraction=0.60, seed=25,
# )

In [ ]:
# --- IG ---
# run_ig_full(
#     valid_subjects=valid_subjects,
#     checkpoint_tag='cdan_16ch_binary',
#     attr_save_path=binary_ig_zero_path,
#     n_classes=2, use_multiclass=False,
#     baseline='zero', n_steps=50,
# )
# run_ig_full(
#     valid_subjects=valid_subjects,
#     checkpoint_tag='multiclass_16ch',
#     attr_save_path=multiclass_ig_zero_path,
#     n_classes=4, use_multiclass=True,
#     baseline='zero', n_steps=50,
# )

### Relevance score - Binary, multi class & top and bottom n channels

#### Binary relevance table

In [ ]:
# Binary relevance score tables - score grouping
binary_lrp_df = build_binary_relevance_table(
    binary_attrs_106, score_groups, ch16_list)

binary_dl_df = build_binary_relevance_table(
    binary_dl_106, score_groups, ch16_list)

binary_shap_df = build_binary_relevance_table(
    binary_shap_106, score_groups, ch16_list)

binary_ig_df = build_binary_relevance_table(
    binary_ig_106, score_groups, ch16_list)

print("Binary relevance tables built: LRP, DeepLIFT, SHAP, IG")


##### table display

In [ ]:
for method_name, df in [('LRP', binary_lrp_df), ('DeepLIFT', binary_dl_df),
                         ('SHAP', binary_shap_df), ('IG', binary_ig_df)]:
    for cls_name in ['left', 'right']:
        print(f"\n{method_name} Binary - Class: {cls_name}")
        pivot = df[df['class'] == cls_name].pivot(
            index='channel', columns='group', values='mean_relevance'
        )[['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']]
        pivot = pivot.reindex(ch16_list)
        pivot.columns = ['High', 'Mid-High', 'Mid-Low', 'Low', 'Neg. Transfer']
        print(pivot.to_string())

#### Multiclass relevance score

In [ ]:
# Multiclass relevance score tables - category grouping
mc_lrp_df = build_multiclass_relevance_table(
    mc_attrs_106, category_groups_106, ch16_list)

mc_dl_df = build_multiclass_relevance_table(
    mc_dl_106, category_groups_106, ch16_list)

mc_shap_df = build_multiclass_relevance_table(
    mc_shap_106, category_groups_106, ch16_list)

mc_ig_df = build_multiclass_relevance_table(
    mc_ig_106, category_groups_106, ch16_list)

print("Multiclass relevance tables built: LRP, DeepLIFT, SHAP, IG")

##### Table display

In [ ]:
for method_name, df in [('LRP', mc_lrp_df), ('DeepLIFT', mc_dl_df),
                         ('SHAP', mc_shap_df), ('IG', mc_ig_df)]:
    for cls_name in ['left', 'right', 'both_fists', 'both_feet']:
        print(f"\n{method_name} Multiclass - Class: {cls_name}")
        pivot = df[df['class'] == cls_name].pivot(
            index='channel', columns='group', values='mean_relevance'
        )[['neither', 'group_a_collapse', 'group_b_flat', 'negative_transfer']]
        pivot = pivot.reindex(ch16_list)
        pivot.columns = ['Neither', 'Group A', 'Group B', 'Neg. Transfer']
        print(pivot.to_string())

#### top/bottom channels

##### Binary top/bottom channels

In [ ]:
for method_name, df in [('LRP', binary_lrp_df), ('DeepLIFT', binary_dl_df),
                         ('SHAP', binary_shap_df), ('IG', binary_ig_df)]:
    print(f"\n{'='*60}")
    print(f"{method_name} Binary - Top/Bottom Channels")
    print(f"{'='*60}")
    get_top_bottom_channels(
        df,
        class_names=['left', 'right'],
        group_order=['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']
    )

##### Multiclass top/bottom channels

In [ ]:
for method_name, df in [('LRP', mc_lrp_df), ('DeepLIFT', mc_dl_df),
                         ('SHAP', mc_shap_df), ('IG', mc_ig_df)]:
    print(f"\n{'='*60}")
    print(f"{method_name} Multiclass - Top/Bottom Channels")
    print(f"{'='*60}")
    get_top_bottom_channels(
        df,
        class_names=['left', 'right', 'both_fists', 'both_feet'],
        group_order=['neither', 'group_a_collapse',
                     'group_b_flat', 'negative_transfer']
    )

### Misclassified data prep

#### Smoke test post refactor
    functions to explainability.py

In [ ]:
print("Smoke test: DeepLIFT binary S006...")
result_dl = run_misclassified_single(
    'S006',
    valid_subjects,
    method='deeplift',
    checkpoint_tag="cdan_16ch_binary",
    n_classes=2,
    use_multiclass=False
)
print(f"n_correct: {result_dl['n_correct']}")
print(f"n_misclassified: {result_dl['n_misclassified']}")
print(f"n_correct + n_misclassified: {result_dl['n_correct'] + result_dl['n_misclassified']}")
print(f"method: {result_dl['method']}")
print(f"confidence shape: {result_dl['misclassified_confidence'].shape}")

In [ ]:
print("Smoke test: IG binary S006...")
result_ig = run_misclassified_single(
    'S006',
    valid_subjects,
    method='ig',
    checkpoint_tag="cdan_16ch_binary",
    n_classes=2,
    use_multiclass=False
)
print(f"n_correct: {result_ig['n_correct']}")
print(f"n_misclassified: {result_ig['n_misclassified']}")
print(f"n_correct + n_misclassified: {result_ig['n_correct'] + result_ig['n_misclassified']}")
print(f"method: {result_ig['method']}")
print(f"confidence shape: {result_ig['misclassified_confidence'].shape}")

print("\nSmoke test: SHAP binary S006...")
result_shap = run_misclassified_single(
    'S006',
    valid_subjects,
    method='shap',
    checkpoint_tag="cdan_16ch_binary",
    n_classes=2,
    use_multiclass=False
)
print(f"n_correct: {result_shap['n_correct']}")
print(f"n_misclassified: {result_shap['n_misclassified']}")
print(f"n_correct + n_misclassified: {result_shap['n_correct'] + result_shap['n_misclassified']}")
print(f"method: {result_shap['method']}")
print(f"confidence shape: {result_shap['misclassified_confidence'].shape}")


#### 106 Subjects misclassified trials files

##### File generated - uncomment for rerun

In [ ]:
# # import time

# binary_misc_path = results_path / "misclassified_attributions" / "binary_cdan_16ch"

# for method in ['deeplift', 'ig', 'shap']:
#     print(f"\n{'='*60}")
#     print(f"Running binary misclassified {method.upper()} for 106 subjects...")
#     print(f"{'='*60}")
#     run_and_save_misclassified(
#         valid_subjects,
#         valid_subjects,
#         save_path=binary_misc_path,
#         method=method,
#         checkpoint_tag="cdan_16ch_binary",
#         n_classes=2,
#         use_multiclass=False,
#         use_16ch=True
#     )

In [ ]:
binary_misc_path = results_path / "misclassified_attributions" / "binary_cdan_16ch"

for method in ['deeplift', 'ig', 'shap']:
    sample = np.load(binary_misc_path / f"S006_misc_{method}.npz",
                     allow_pickle=True)
    n_correct = int(sample['n_correct'])
    n_misc = int(sample['n_misclassified'])
    conf_shape = sample['misclassified_confidence'].shape
    print(f"{method.upper()}: n_correct={n_correct}, "
          f"n_misclassified={n_misc}, "
          f"total={n_correct+n_misc}, "
          f"confidence_shape={conf_shape}")

#### Missclassification files for multiclass

##### File generated - uncomment for rerun

In [ ]:
# multiclass_misc_path = results_path / "misclassified_attributions" / "multiclass_16ch"

# for method in ['deeplift', 'ig', 'shap']:
#     print(f"\n{'='*60}")
#     print(f"Running multiclass misclassified {method.upper()} for 106 subjects...")
#     print(f"{'='*60}")
#     run_and_save_misclassified(
#         valid_subjects,
#         valid_subjects,
#         save_path=multiclass_misc_path,
#         method=method,
#         checkpoint_tag="multiclass_16ch",
#         n_classes=4,
#         use_multiclass=True,
#         use_16ch=True
#     )

In [ ]:
multiclass_misc_path = results_path / "misclassified_attributions" / "multiclass_16ch"

for method in ['deeplift', 'ig', 'shap']:
    sample = np.load(multiclass_misc_path / f"S006_misc_{method}.npz",
                     allow_pickle=True)
    n_correct = int(sample['n_correct'])
    n_misc = int(sample['n_misclassified'])
    conf_shape = sample['misclassified_confidence'].shape
    n_classes = int(sample['n_classes'])
    print(f"{method.upper()}: n_correct={n_correct}, "
          f"n_misclassified={n_misc}, "
          f"total={n_correct+n_misc}, "
          f"n_classes={n_classes}, "
          f"confidence_shape={conf_shape}")

### Misclassified analysis

In [ ]:
misc_path = results_path / "misclassified_attributions" / "binary_cdan_16ch"

sample = np.load(misc_path / "S006_misc_lrp.npz", allow_pickle=True)
print("Keys:", list(sample.keys()))
print(f"n_correct: {sample['n_correct']}")
print(f"n_misclassified: {sample['n_misclassified']}")
print(f"correct_mean_class_0 shape: {sample['correct_mean_class_0'].shape}")
print(f"misclassified_mean_class_0 shape: {sample['misclassified_mean_class_0'].shape}")
print(f"misclassified_label_pairs shape: {sample['misclassified_label_pairs'].shape}")
print(f"misclassified_confidence shape: {sample['misclassified_confidence'].shape}")
print(f"misclassified_label_pairs sample:\n{sample['misclassified_label_pairs'][:5]}")

#### Binary missclassification

In [ ]:
def load_misclassified_binary(subject_id, method='lrp'):
    """Load misclassified attribution file for one subject."""
    f = np.load(binary_misc_path / f"{subject_id}_misc_{method}.npz",
                allow_pickle=True)
    return f

def compute_misc_analysis_single(subject_id, method='lrp'):
    """
    For one subject compute:
    1. Mean attribution difference: correct - misclassified per class
    2. Error direction counts from label pairs
    3. Mean confidence for misclassified trials
    """
    f = np.load(binary_misc_path / f"{subject_id}_misc_{method}.npz",
                allow_pickle=True)

    n_correct = int(f['n_correct'])
    n_misc = int(f['n_misclassified'])

    if n_misc == 0:
        return None

    # 1. Attribution difference per class
    attr_diff = {}
    for cls in range(2):
        correct_attr = np.array(f[f'correct_mean_class_{cls}'])
        misc_attr = np.array(f[f'misclassified_mean_class_{cls}'])

        # Skip if empty
        if correct_attr.ndim != 2 or misc_attr.ndim != 2:
            attr_diff[cls] = np.zeros(16)
            continue

        correct_ch = np.mean(correct_attr, axis=1)
        misc_ch = np.mean(misc_attr, axis=1)
        attr_diff[cls] = correct_ch - misc_ch

    # 2. Error direction counts
    label_pairs = f['misclassified_label_pairs']
    error_counts = {}
    for true_label, pred_label in label_pairs:
        key = f"true{int(true_label)}_pred{int(pred_label)}"
        error_counts[key] = error_counts.get(key, 0) + 1

    # 3. Mean confidence
    confidence = np.array(f['misclassified_confidence'])
    mean_conf_misc = float(np.mean(np.max(confidence, axis=1)))

    return {
        'n_correct': n_correct,
        'n_misclassified': n_misc,
        'attr_diff': attr_diff,
        'error_counts': error_counts,
        'mean_confidence_misclassified': mean_conf_misc,
    }


In [ ]:
# Test on S006 LRP
print("Testing S006 LRP...")
result = compute_misc_analysis_single('S006', method='lrp')
print(f"n_correct: {result['n_correct']}")
print(f"n_misclassified: {result['n_misclassified']}")
print(f"Error counts: {result['error_counts']}")
print(f"Mean confidence misclassified: {result['mean_confidence_misclassified']:.4f}")
print(f"Attr diff class 0 (top 3 channels):")
diff_c0 = result['attr_diff'][0]
top3_idx = np.argsort(np.abs(diff_c0))[::-1][:3]
for idx in top3_idx:
    print(f"  {ch16_list[idx]}: {diff_c0[idx]:.6f}")

##### full subject

In [ ]:

# Rerun
print("Running binary misclassification analysis...")
binary_misc_results = {}
skipped = []

for subj in valid_subjects:
    result = compute_misc_analysis_single(subj, method='lrp')
    if result is None:
        skipped.append(subj)
        continue
    binary_misc_results[subj] = result

print(f"Subjects with misclassified trials: {len(binary_misc_results)}")
print(f"Subjects with zero misclassified: {len(skipped)}")
print(f"Skipped: {skipped}")

# Population level error direction
total_errors = {'true0_pred1': 0, 'true1_pred0': 0}
for subj, res in binary_misc_results.items():
    for key, count in res['error_counts'].items():
        total_errors[key] = total_errors.get(key, 0) + count

print(f"\nPopulation level error direction (binary, 106 subjects):")
print(f"  Left predicted as right (true0_pred1): {total_errors.get('true0_pred1', 0)}")
print(f"  Right predicted as left (true1_pred0): {total_errors.get('true1_pred0', 0)}")
total = sum(total_errors.values())
print(f"  Total errors: {total}")
print(f"  Left-as-right rate: {total_errors.get('true0_pred1', 0)/total*100:.1f}%")
print(f"  Right-as-left rate: {total_errors.get('true1_pred0', 0)/total*100:.1f}%")

conf_values = [res['mean_confidence_misclassified']
               for res in binary_misc_results.values()]
print(f"\nMean confidence on misclassified trials: {np.mean(conf_values):.4f}")
print(f"Std: {np.std(conf_values):.4f}")
print(f"Min: {np.min(conf_values):.4f}, Max: {np.max(conf_values):.4f}")

In [ ]:
# Reprint with full context
total_trials = sum(res['n_correct'] + res['n_misclassified']
                   for res in binary_misc_results.values())
total_correct = sum(res['n_correct'] for res in binary_misc_results.values())
total_misc = sum(res['n_misclassified'] for res in binary_misc_results.values())

print("Binary misclassification summary (LRP, 106 subjects)")
print("="*60)
print(f"Total trials across 104 subjects with errors: {total_trials}")
print(f"Correctly classified: {total_correct} "
      f"({total_correct/total_trials*100:.1f}%)")
print(f"Misclassified: {total_misc} "
      f"({total_misc/total_trials*100:.1f}%)")
print(f"\nOf the {total_misc} misclassified trials:")
print(f"  Left predicted as right: "
      f"{total_errors.get('true0_pred1', 0)} "
      f"({total_errors.get('true0_pred1', 0)/total_misc*100:.1f}% of errors)")
print(f"  Right predicted as left: "
      f"{total_errors.get('true1_pred0', 0)} "
      f"({total_errors.get('true1_pred0', 0)/total_misc*100:.1f}% of errors)")
print(f"\nMean model confidence on misclassified trials: "
      f"{np.mean(conf_values):.4f}")

print(f"\nSubjects with zero misclassified trials: 2 (S007, S062)")

In [ ]:
# Population level attribution difference: correct vs misclassified
# Average attr_diff across all subjects for each class
attr_diff_pop = {0: np.zeros(16), 1: np.zeros(16)}
count_per_class = {0: 0, 1: 0}

for subj, res in binary_misc_results.items():
    for cls in range(2):
        if np.any(res['attr_diff'][cls] != 0):
            attr_diff_pop[cls] += res['attr_diff'][cls]
            count_per_class[cls] += 1

# Average
for cls in range(2):
    if count_per_class[cls] > 0:
        attr_diff_pop[cls] /= count_per_class[cls]

# Print top channels where correct and misclassified differ most
print("Population mean attribution difference (correct - misclassified)")
print("Positive = channel more relevant in correct trials")
print("Negative = channel more relevant in misclassified trials")

for cls, cls_name in [(0, 'left'), (1, 'right')]:
    print(f"\nClass {cls_name} (n={count_per_class[cls]} subjects):")
    diff = attr_diff_pop[cls]
    print(f"{'Channel':<8}{'Diff':>10}")
    print("-"*20)
    # Sort by absolute difference
    sorted_idx = np.argsort(np.abs(diff))[::-1]
    for idx in sorted_idx[:8]:
        print(f"{ch16_list[idx]:<8}{diff[idx]:>10.6f}")

#### Missclassified multiclass

In [ ]:
sample_mc = np.load(multiclass_misc_path / "S006_misc_lrp.npz", allow_pickle=True)
print("Keys:", list(sample_mc.keys()))
print(f"n_correct: {sample_mc['n_correct']}")
print(f"n_misclassified: {sample_mc['n_misclassified']}")
print(f"n_classes: {sample_mc['n_classes']}")
print(f"misclassified_label_pairs shape: {sample_mc['misclassified_label_pairs'].shape}")
print(f"misclassified_label_pairs sample:\n{sample_mc['misclassified_label_pairs'][:5]}")
print(f"misclassified_confidence shape: {sample_mc['misclassified_confidence'].shape}")

In [ ]:
class_names_mc = {0: 'left', 1: 'right', 2: 'both_fists', 3: 'both_feet'}

def compute_misc_analysis_multiclass(subject_id, method='lrp'):
    """
    Multiclass misclassification analysis for one subject.
    1. Error direction counts from label pairs
    2. Attribution difference correct vs misclassified per class
    3. Mean confidence on misclassified trials
    """
    f = np.load(multiclass_misc_path / f"{subject_id}_misc_{method}.npz",
                allow_pickle=True)

    n_correct = int(f['n_correct'])
    n_misc = int(f['n_misclassified'])

    if n_misc == 0:
        return None

    # 1. Error direction counts
    label_pairs = f['misclassified_label_pairs']
    error_counts = {}
    for true_label, pred_label in label_pairs:
        true_name = class_names_mc[int(true_label)]
        pred_name = class_names_mc[int(pred_label)]
        key = f"{true_name}_as_{pred_name}"
        error_counts[key] = error_counts.get(key, 0) + 1

    # 2. Attribution difference per class
    attr_diff = {}
    for cls in range(4):
        correct_attr = np.array(f[f'correct_mean_class_{cls}'])
        misc_attr = np.array(f[f'misclassified_mean_class_{cls}'])
        if correct_attr.ndim != 2 or misc_attr.ndim != 2:
            attr_diff[cls] = np.zeros(16)
            continue
        correct_ch = np.mean(correct_attr, axis=1)
        misc_ch = np.mean(misc_attr, axis=1)
        attr_diff[cls] = correct_ch - misc_ch

    # 3. Mean confidence
    confidence = np.array(f['misclassified_confidence'])
    mean_conf = float(np.mean(np.max(confidence, axis=1)))

    return {
        'n_correct': n_correct,
        'n_misclassified': n_misc,
        'error_counts': error_counts,
        'attr_diff': attr_diff,
        'mean_confidence_misclassified': mean_conf
    }


In [ ]:
# Test on S006
print("Testing S006 multiclass LRP...")
result_mc = compute_misc_analysis_multiclass('S006', method='lrp')
print(f"n_correct: {result_mc['n_correct']}")
print(f"n_misclassified: {result_mc['n_misclassified']}")
print(f"Error counts: {result_mc['error_counts']}")
print(f"Mean confidence misclassified: {result_mc['mean_confidence_misclassified']:.4f}")

##### full subject

In [ ]:
print("Running multiclass misclassification analysis...")
mc_misc_results = {}
mc_skipped = []

for subj in valid_subjects:
    result = compute_misc_analysis_multiclass(subj, method='lrp')
    if result is None:
        mc_skipped.append(subj)
        continue
    mc_misc_results[subj] = result

print(f"Subjects with misclassified trials: {len(mc_misc_results)}")
print(f"Subjects with zero misclassified: {len(mc_skipped)}")
print(f"Skipped: {mc_skipped}")

# Population level totals
total_trials_mc = sum(res['n_correct'] + res['n_misclassified']
                      for res in mc_misc_results.values())
total_correct_mc = sum(res['n_correct'] for res in mc_misc_results.values())
total_misc_mc = sum(res['n_misclassified'] for res in mc_misc_results.values())

print(f"\nMulticlass misclassification summary (LRP, 106 subjects)")
print("="*60)
print(f"Total trials: {total_trials_mc}")
print(f"Correctly classified: {total_correct_mc} "
      f"({total_correct_mc/total_trials_mc*100:.1f}%)")
print(f"Misclassified: {total_misc_mc} "
      f"({total_misc_mc/total_trials_mc*100:.1f}%)")

# Population level error direction
all_error_counts = {}
for res in mc_misc_results.values():
    for key, count in res['error_counts'].items():
        all_error_counts[key] = all_error_counts.get(key, 0) + count

print(f"\nError direction counts (all {total_misc_mc} misclassified trials):")
for key, count in sorted(all_error_counts.items(),
                          key=lambda x: x[1], reverse=True):
    pct = count/total_misc_mc*100
    print(f"  {key}: {count} ({pct:.1f}%)")

# Mean confidence
conf_mc = [res['mean_confidence_misclassified']
           for res in mc_misc_results.values()]
print(f"\nMean confidence on misclassified trials: {np.mean(conf_mc):.4f}")
print(f"Std: {np.std(conf_mc):.4f}")

##### # Population mean attribution difference multiclass


In [ ]:
# Population mean attribution difference multiclass
attr_diff_pop_mc = {cls: np.zeros(16) for cls in range(4)}
count_per_class_mc = {cls: 0 for cls in range(4)}

for subj, res in mc_misc_results.items():
    for cls in range(4):
        if np.any(res['attr_diff'][cls] != 0):
            attr_diff_pop_mc[cls] += res['attr_diff'][cls]
            count_per_class_mc[cls] += 1

for cls in range(4):
    if count_per_class_mc[cls] > 0:
        attr_diff_pop_mc[cls] /= count_per_class_mc[cls]

print("Population mean attribution difference multiclass")
print("Positive = channel more relevant in correct trials")
print("Negative = channel more relevant in misclassified trials")

for cls in range(4):
    cls_name = class_names_mc[cls]
    print(f"\nClass {cls_name} (n={count_per_class_mc[cls]} subjects):")
    diff = attr_diff_pop_mc[cls]
    print(f"{'Channel':<8}{'Diff':>10}")
    print("-"*20)
    sorted_idx = np.argsort(np.abs(diff))[::-1]
    for idx in sorted_idx[:6]:
        print(f"{ch16_list[idx]:<8}{diff[idx]:>10.6f}")

## last pending analysis - 15 july

### wrong hemisphere check

In [ ]:
# Run attribution difference for all four methods
for method in ['lrp', 'deeplift', 'ig', 'shap']:
    print(f"\n{'='*50}")
    print(f"Method: {method.upper()}")
    print(f"{'='*50}")

    results = {}
    for subj in valid_subjects:
        result = compute_misc_analysis_single(subj, method=method)
        if result is None:
            continue
        results[subj] = result

    # Population mean attribution difference
    attr_diff_pop = {0: np.zeros(16), 1: np.zeros(16)}
    count_per_class = {0: 0, 1: 0}

    for subj, res in results.items():
        for cls in range(2):
            if np.any(res['attr_diff'][cls] != 0):
                attr_diff_pop[cls] += res['attr_diff'][cls]
                count_per_class[cls] += 1

    for cls in range(2):
        if count_per_class[cls] > 0:
            attr_diff_pop[cls] /= count_per_class[cls]

    for cls, cls_name in [(0, 'left'), (1, 'right')]:
        print(f"\nClass {cls_name} (n={count_per_class[cls]}):")
        diff = attr_diff_pop[cls]
        sorted_idx = np.argsort(np.abs(diff))[::-1]
        for idx in sorted_idx[:6]:
            print(f"  {ch16_list[idx]:<8}{diff[idx]:>10.6f}")

In [ ]:
# BINARY: all 16 channels, all four methods
print("Generating full 16-channel attribution difference table")
print("Binary task all four methods")

binary_rows = []

for method in ['lrp', 'deeplift', 'ig', 'shap']:
    results = {}
    for subj in valid_subjects:
        result = compute_misc_analysis_single(subj, method=method)
        if result is None:
            continue
        results[subj] = result

    attr_diff_pop = {0: np.zeros(16), 1: np.zeros(16)}
    count_per_class = {0: 0, 1: 0}

    for subj, res in results.items():
        for cls in range(2):
            if np.any(res['attr_diff'][cls] != 0):
                attr_diff_pop[cls] += res['attr_diff'][cls]
                count_per_class[cls] += 1

    for cls in range(2):
        if count_per_class[cls] > 0:
            attr_diff_pop[cls] /= count_per_class[cls]

    for cls, cls_name in [(0, 'left'), (1, 'right')]:
        for ch_i, ch_name in enumerate(ch16_list):
            binary_rows.append({
                'method': method.upper(),
                'class': cls_name,
                'channel': ch_name,
                'attr_diff': round(float(attr_diff_pop[cls][ch_i]), 6),
                'n_subjects': count_per_class[cls]
            })

df_binary_full = pd.DataFrame(binary_rows)

In [ ]:
# Binary pivoted tables
print("BINARY Attribution difference, correct minus misclassified")
print("Positive = channel more relevant in correct trials, \nNegative = channel more relevant in misclassified trials")
print()

for cls_name in ['left', 'right']:
    df_cls = df_binary_full[df_binary_full['class'] == cls_name]
    df_pivot = df_cls.pivot(
        index='channel', columns='method', values='attr_diff'
    )[['LRP', 'DEEPLIFT', 'IG', 'SHAP']]
    df_pivot = df_pivot.reindex(ch16_list)
    print(f"Binary: {cls_name} class")
    print(df_pivot.to_string())
    print()

In [ ]:
# MULTICLASS: all 16 channels, all four methods
print("\nMulticlass task all four methods")

mc_rows = []
class_names_list = ['left', 'right', 'both_fists', 'both_feet']

for method in ['lrp', 'deeplift', 'ig', 'shap']:
    results = {}
    for subj in valid_subjects:
        result = compute_misc_analysis_multiclass(subj, method=method)
        if result is None:
            continue
        results[subj] = result

    attr_diff_pop = {cls: np.zeros(16) for cls in range(4)}
    count_per_class = {cls: 0 for cls in range(4)}

    for subj, res in results.items():
        for cls in range(4):
            if np.any(res['attr_diff'][cls] != 0):
                attr_diff_pop[cls] += res['attr_diff'][cls]
                count_per_class[cls] += 1

    for cls in range(4):
        if count_per_class[cls] > 0:
            attr_diff_pop[cls] /= count_per_class[cls]

    for cls, cls_name in enumerate(class_names_list):
        for ch_i, ch_name in enumerate(ch16_list):
            mc_rows.append({
                'method': method.upper(),
                'class': cls_name,
                'channel': ch_name,
                'attr_diff': round(float(attr_diff_pop[cls][ch_i]), 6),
                'n_subjects': count_per_class[cls]
            })

df_mc_full = pd.DataFrame(mc_rows)

In [ ]:
# Multiclass pivoted tables
print("MULTICLASS Attribution difference, correct minus misclassified")
print("Positive = channel more relevant in correct trials, \nNegative = channel more relevant in misclassified trials")
print()

for cls_name in ['left', 'right', 'both_fists', 'both_feet']:
    df_cls = df_mc_full[df_mc_full['class'] == cls_name]
    df_pivot = df_cls.pivot(
        index='channel', columns='method', values='attr_diff'
    )[['LRP', 'DEEPLIFT', 'IG', 'SHAP']]
    df_pivot = df_pivot.reindex(ch16_list)
    print(f"Multiclass: {cls_name} class")
    print(df_pivot.to_string())
    print()

In [ ]:
# Save both to CSV: base df
binary_export = results_path / 'misc_attr_diff_binary_all_methods.csv'
mc_export = results_path / 'misc_attr_diff_multiclass_all_methods.csv'

df_binary_full.to_csv(binary_export, index=False)
df_mc_full.to_csv(mc_export, index=False)

print(f"\nSaved binary table: {binary_export}")
print(f"Saved multiclass table: {mc_export}")
print(f"Binary rows: {len(df_binary_full)}")
print(f"Multiclass rows: {len(df_mc_full)}")

# Preview key channels for confirmation
print("\nBinary LRP left class preview:")
print(df_binary_full[
    (df_binary_full['method']=='LRP') &
    (df_binary_full['class']=='left')
][['channel','attr_diff']].to_string(index=False))

In [ ]:
# Save pivoted versions
for cls_name in ['left', 'right']:
    df_cls = df_binary_full[df_binary_full['class'] == cls_name]
    df_pivot = df_cls.pivot(
        index='channel', columns='method', values='attr_diff'
    )[['LRP', 'DEEPLIFT', 'IG', 'SHAP']]
    df_pivot = df_pivot.reindex(ch16_list)
    export = results_path / f'misc_attr_diff_binary_{cls_name}_pivot.csv'
    df_pivot.to_csv(export)

for cls_name in ['left', 'right', 'both_fists', 'both_feet']:
    df_cls = df_mc_full[df_mc_full['class'] == cls_name]
    df_pivot = df_cls.pivot(
        index='channel', columns='method', values='attr_diff'
    )[['LRP', 'DEEPLIFT', 'IG', 'SHAP']]
    df_pivot = df_pivot.reindex(ch16_list)
    export = results_path / f'misc_attr_diff_mc_{cls_name}_pivot.csv'
    df_pivot.to_csv(export)

print("All pivot CSVs saved.")

### subject level deviation score correlated with kappa - shap

In [ ]:
with open(results_path / 'loso_results_cdan_16ch_RETRAIN.json') as f:
    binary_results = json.load(f)

binary_kappa = {
    subj: binary_results[subj]['kappa_stopping']['final_kappa']
    for subj in valid_subjects
}

print(f"Loaded: {len(binary_kappa)} subjects")
print(f"Kappa range: {min(binary_kappa.values()):.4f} to {max(binary_kappa.values()):.4f}")

In [ ]:
# Compute mean absolute SHAP value per subject as deviation score
shap_deviation = {}
for subj in valid_subjects:
    # Mean absolute attribution across all channels and timepoints
    # Use kappa stopping, both classes averaged
    class_0 = np.array(binary_shap_106[subj]['kappa']['mean_by_class'][0])
    class_1 = np.array(binary_shap_106[subj]['kappa']['mean_by_class'][1])
    mean_abs = float(np.mean(np.abs(class_0) + np.abs(class_1)) / 2)
    shap_deviation[subj] = mean_abs

# Correlate with kappa
deviation_values = [shap_deviation[s] for s in valid_subjects]
kappa_values_list = [binary_kappa[s] for s in valid_subjects]

correlation = np.corrcoef(deviation_values, kappa_values_list)[0, 1]
print(f"SHAP deviation vs kappa correlation: r={correlation:.4f}")

# Print top and bottom 5 by deviation
sorted_by_deviation = sorted(shap_deviation.items(),
                              key=lambda x: x[1], reverse=True)
print("\nTop 5 highest deviation (most different from source pool):")
for subj, val in sorted_by_deviation[:5]:
    print(f"  {subj}: deviation={val:.6f}, kappa={binary_kappa[subj]:.4f}")

print("\nBottom 5 lowest deviation (most similar to source pool):")
for subj, val in sorted_by_deviation[-5:]:
    print(f"  {subj}: deviation={val:.6f}, kappa={binary_kappa[subj]:.4f}")

### Cosine simialrity for atrribution maps

In [ ]:
def compute_cosine_consistency_all(attrs_106, category_groups, valid_subjects,
                                    stopping='kappa', n_classes=2,
                                    method_name=""):
    """
    Compute mean pairwise cosine similarity within each group.
    Works for any method and any number of classes.
    """
    class_names_map = {
        0: 'left', 1: 'right', 2: 'both_fists', 3: 'both_feet'
    }
    groups = ['neither', 'group_a_collapse',
              'group_b_flat', 'negative_transfer']

    print(f"\nCosine similarity | {method_name} | category grouping")
    print("-"*60)

    for cls in range(n_classes):
        cls_name = class_names_map[cls]
        print(f"\nClass {cls_name}:")
        for group in groups:
            group_subjects = [s for s in valid_subjects
                            if category_groups[s] == group]

            maps = []
            for subj in group_subjects:
                attr = np.array(
                    attrs_106[subj][stopping]['mean_by_class'][cls])
                if attr.ndim == 2:
                    maps.append(attr.flatten())

            if len(maps) < 2:
                print(f"  {group}: insufficient subjects")
                continue

            maps_arr = np.array(maps)
            sim_matrix = cosine_similarity(maps_arr)
            n = len(maps_arr)
            upper_idx = np.triu_indices(n, k=1)
            pairwise_sims = sim_matrix[upper_idx]

            print(f"  {group}: mean={np.mean(pairwise_sims):.4f} "
                  f"(std={np.std(pairwise_sims):.4f}, "
                  f"n={n}, pairs={len(pairwise_sims)})")

# Run all methods binary
print("="*60)
print("BINARY - Cosine similarity all methods")
print("="*60)

compute_cosine_consistency_all(
    binary_attrs_106, category_groups_106, valid_subjects,
    n_classes=2, method_name="LRP binary")

compute_cosine_consistency_all(
    binary_dl_106, category_groups_106, valid_subjects,
    n_classes=2, method_name="DeepLIFT binary")

compute_cosine_consistency_all(
    binary_shap_106, category_groups_106, valid_subjects,
    n_classes=2, method_name="SHAP binary")

compute_cosine_consistency_all(
    binary_ig_106, category_groups_106, valid_subjects,
    n_classes=2, method_name="IG binary")

# Run all methods multiclass
print("\n" + "="*60)
print("MULTICLASS - Cosine similarity all methods")
print("="*60)

compute_cosine_consistency_all(
    mc_attrs_106, category_groups_106, valid_subjects,
    n_classes=4, method_name="LRP multiclass")

compute_cosine_consistency_all(
    mc_dl_106, category_groups_106, valid_subjects,
    n_classes=4, method_name="DeepLIFT multiclass")

compute_cosine_consistency_all(
    mc_shap_106, category_groups_106, valid_subjects,
    n_classes=4, method_name="SHAP multiclass")

compute_cosine_consistency_all(
    mc_ig_106, category_groups_106, valid_subjects,
    n_classes=4, method_name="IG multiclass")

### Hypothesis Testing

#### Hypothesis: Mann-Whitney U test on mean ipsilateral relevance, high vs low performance groups, combined left and right classes


In [ ]:
base_path = '..'
lrp_binary_path = os.path.join(base_path, 'results', 'lrp_attributions', 'binary_cdan_16ch')

# Check structure of one file
test_file = os.path.join(lrp_binary_path, 'S001_lrp.npz')
data = np.load(test_file, allow_pickle=True)

print("Keys in LRP npz file:")
for k in data.keys():
    print(f"  {k}: shape {data[k].shape if hasattr(data[k], 'shape') else type(data[k])}")

In [ ]:
ch_idx = {ch: i for i, ch in enumerate(ch16_list)}

def get_mean_attr(subj, cls_idx, criterion='kappa'):
    data = binary_attrs_106[subj][criterion]['mean_by_class'][cls_idx]
    if data is None:
        return None
    return data.mean(axis=1)

def mean_ipsi(subj, cls_idx, ipsi_chs):
    arr = get_mean_attr(subj, cls_idx)
    if arr is None:
        return None
    return np.mean([arr[ch_idx[ch]] for ch in ipsi_chs])

In [ ]:
group_order = ['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']
key_chs = ['C3', 'CP3', 'C4', 'CP4', 'CPz']

print("Score group counts:")
from collections import Counter
print(dict(Counter(score_groups.values())))
print()

for cls_idx, cls_name in [(0, 'LEFT'), (1, 'RIGHT')]:
    print(f"\n{cls_name} class — mean relevance by score group:")
    header = f"{'Group':<22}" + "".join(f"{ch:>10}" for ch in key_chs)
    print(header)
    print("-" * (22 + 10 * len(key_chs)))
    for grp in group_order:
        subjs = [s for s, g in score_groups.items() if g == grp]
        attrs = []
        for s in subjs:
            if s in binary_attrs_106:
                arr = get_mean_attr(s, cls_idx)
                if arr is not None:
                    attrs.append(arr)
        if attrs:
            mean_attr = np.mean(attrs, axis=0)
            row = f"{grp:<22}"
            for ch in key_chs:
                row += f"{mean_attr[ch_idx[ch]]:>10.6f}"
            print(row)

# Mann-Whitney: high vs low, ipsilateral relevance
print("\n")
print("-"* 70)
print("Mann-Whitney U: high vs low, mean ipsilateral relevance")

high_subjs = [s for s, g in score_groups.items() if g == 'high']
low_subjs  = [s for s, g in score_groups.items() if g == 'low']

for cls_idx, cls_name, ipsi_chs in [
    (0, 'LEFT',  ['C3', 'CP3']),
    (1, 'RIGHT', ['C4', 'CP4'])
]:
    high_vals = [v for s in high_subjs
                 if s in binary_attrs_106
                 and (v := mean_ipsi(s, cls_idx, ipsi_chs)) is not None]
    low_vals  = [v for s in low_subjs
                 if s in binary_attrs_106
                 and (v := mean_ipsi(s, cls_idx, ipsi_chs)) is not None]
    u, p = mannwhitneyu(high_vals, low_vals, alternative='greater')
    print(f"{cls_name}: U={u:.1f}, p={p:.4f}, "
          f"high n={len(high_vals)}, low n={len(low_vals)}")

In [ ]:
category_order = ['neither', 'group_a_collapse', 'group_b_flat', 'negative_transfer']
key_chs = ['C3', 'CP3', 'C4', 'CP4', 'CPz']

print("Category group counts:")
from collections import Counter
print(dict(Counter(category_groups_106.values())))
print()

for cls_idx, cls_name in [(0, 'LEFT'), (1, 'RIGHT')]:
    print(f"\n{cls_name} class — mean LRP relevance by category group:")
    header = f"{'Group':<22}" + "".join(f"{ch:>10}" for ch in key_chs)
    print(header)
    print("-" * (22 + 10 * len(key_chs)))
    for grp in category_order:
        subjs = [s for s, g in category_groups_106.items() if g == grp]
        attrs = []
        for s in subjs:
            if s in binary_attrs_106:
                arr = get_mean_attr(s, cls_idx)
                if arr is not None:
                    attrs.append(arr)
        if attrs:
            mean_attr = np.mean(attrs, axis=0)
            row = f"{grp:<22}"
            for ch in key_chs:
                row += f"{mean_attr[ch_idx[ch]]:>10.6f}"
            print(row)

#### Hypothesis Test: Mean Ipsilateral Relevance, High vs Low Performance Groups"
attribution strength across performance groups (Combined Ipsilateral Score)

Granular version above is reported and this is skipped

In [ ]:
# Hypothesis Test: Attribution strength - High vs Low group
# Mann-Whitney U test comparing mean ipsilateral relevance

def get_ipsi_relevance(subjects, lrp_data):
    scores = []
    for subject in subjects:
        if subject in lrp_data:
            left_map = lrp_data[subject]['kappa']['mean_by_class'][0]
            right_map = lrp_data[subject]['kappa']['mean_by_class'][1]
            if left_map is not None and right_map is not None:
                left_ch = left_map.mean(axis=1)
                right_ch = right_map.mean(axis=1)
                ipsi_left = (left_ch[0] + left_ch[5]) / 2   # C3, CP3
                ipsi_right = (right_ch[2] + right_ch[7]) / 2 # C4, CP4
                scores.append((ipsi_left + ipsi_right) / 2)
    return np.array(scores)

high_subjects = [s for s, g in score_groups.items() if g == 'high']
low_subjects = [s for s, g in score_groups.items() if g == 'low']

high_ipsi = get_ipsi_relevance(high_subjects, binary_attrs_106)
low_ipsi = get_ipsi_relevance(low_subjects, binary_attrs_106)

stat, p = stats.mannwhitneyu(high_ipsi, low_ipsi, alternative='greater')

print(f"High group n={len(high_ipsi)}, mean ipsilateral relevance: {high_ipsi.mean():.6f}")
print(f"Low group n={len(low_ipsi)}, mean ipsilateral relevance: {low_ipsi.mean():.6f}")
print(f"Mann-Whitney U: stat={stat:.1f}, p={p:.6f}")

In [ ]:
neg_transfer_subjs = [s for s, g in category_groups_106.items()
                      if g == 'negative_transfer']
print("Negative transfer subjects:", neg_transfer_subjs)
print()
for s in neg_transfer_subjs:
    if s in mc_attrs_106:
        arr = mc_attrs_106[s]['kappa']['mean_by_class'][2]
        print(f"{s} both_fists attribution: "
              f"{'None' if arr is None else arr.shape}")
    else:
        print(f"{s}: not in mc_attrs_106")

### Paired Test

#### Load Files

In [ ]:
def load_json(name):
    with open(results_path / name, 'r') as f:
        return json.load(f)

res = {
    'baseline_8ch':  load_json('loso_results_baseline.json'),
    'baseline_16ch': load_json('loso_results_baseline_16ch.json'),
    'cdan_8ch':      load_json('loso_results_cdan.json'),
    'cdan_16ch':     load_json('loso_results_cdan_16ch_RETRAIN.json'),  # thesis values, used for XAI
}

def entry(model, subject, criterion='kappa_stopping'):
    # baseline files hold one run per subject, CDAN files one run per stopping criterion
    r = res[model][subject]
    return r if model.startswith('baseline') else r[criterion]

def kappas(model, criterion='kappa_stopping'):
    return np.array([entry(model, s, criterion)['final_kappa'] for s in valid_subjects])

def kappa_at_epoch(model, epoch, criterion='kappa_stopping'):
    return np.array([entry(model, s, criterion)['log']['kappa'][epoch - 1] for s in valid_subjects])

for m in res:
    print(f"{m}: {len(res[m])} subjects, mean kappa {kappas(m).mean():.4f}")

#### Paired Test

In [ ]:
# paired comparisons on the 106 LOSO subjects
rng = np.random.default_rng(0)

def paired_test(a, b, n_boot=10000):
    d = a - b
    w = stats.wilcoxon(a, b)
    t = stats.ttest_rel(a, b)
    boot = rng.choice(d, (n_boot, len(d)), replace=True).mean(axis=1)
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
    nz = d[d != 0]
    ranks = stats.rankdata(np.abs(nz))
    return {
        'mean_a': a.mean(), 'mean_b': b.mean(), 'mean_diff': d.mean(),
        'ci_low': ci_low, 'ci_high': ci_high, 'median_diff': np.median(d),
        'wilcoxon_W': w.statistic, 'wilcoxon_p': w.pvalue,
        't': t.statistic, 't_p': t.pvalue,
        'd_z': d.mean() / d.std(ddof=1),
        'rank_biserial': (ranks[nz > 0].sum() - ranks[nz < 0].sum()) / ranks.sum(),
        'better': int((d > 0).sum()), 'worse': int((d < 0).sum()), 'tie': int((d == 0).sum()),
    }

comparisons = [
    ('CDAN 16ch (kappa stop) vs baseline 16ch', kappas('cdan_16ch'), kappas('baseline_16ch')),
    ('CDAN 16ch (composite stop) vs baseline 16ch',
     kappas('cdan_16ch', 'composite_stopping'), kappas('baseline_16ch')),
    ('baseline 16ch vs baseline 8ch', kappas('baseline_16ch'), kappas('baseline_8ch')),
    ('CDAN 16ch vs baseline 8ch', kappas('cdan_16ch'), kappas('baseline_8ch')),
    ('CDAN 8ch vs baseline 8ch', kappas('cdan_8ch'), kappas('baseline_8ch')),
]
for ep in (10, 20, 30):
    comparisons.append((f'fixed epoch {ep}: CDAN 16ch vs baseline 16ch',
                        kappa_at_epoch('cdan_16ch', ep), kappa_at_epoch('baseline_16ch', ep)))

paired_df = pd.DataFrame([{'comparison': n, **paired_test(a, b)} for n, a, b in comparisons]).round(4)
paired_df.to_csv(results_path / 'v2_paired_tests.csv', index=False)
paired_df

In [ ]:
# new Table 2 row
b16 = kappas('baseline_16ch')
print(f"Baseline 16ch: mean {b16.mean():.4f}, median {np.median(b16):.4f}, "
      f"above chance (kappa > 0.1) {np.sum(b16 > 0.1)}/106\n")

# reported kappa = best per-epoch kappa on the target subject
for m in res:
    same = sum(np.isclose(entry(m, s)['final_kappa'], max(entry(m, s)['log']['kappa']))
               for s in valid_subjects)
    print(f"{m}: reported kappa equals best per-epoch kappa for {same}/106 subjects")

# best epoch inside the 10-epoch warmup, i.e. before any adversarial training
for crit in ('kappa_stopping', 'composite_stopping'):
    best = np.array([entry('cdan_16ch', s, crit)['log']['best_epoch'] for s in valid_subjects])
    print(f"CDAN 16ch ({crit}): best epoch within warmup for {np.sum(best <= 10)}/106 subjects")

# mean kappa without best-epoch selection
for ep in (10, 20, 30):
    print(f"epoch {ep}: CDAN 16ch {kappa_at_epoch('cdan_16ch', ep).mean():.4f} | "
          f"baseline 16ch {kappa_at_epoch('baseline_16ch', ep).mean():.4f}")

In [ ]:
# per-subject kappa, CDAN 16ch vs no-adaptation baseline 16ch
CM = 1 / 2.54
figures_path = Path("../results/figures/")
style = {'font.size': 8, 'axes.labelsize': 8, 'xtick.labelsize': 7,
         'ytick.labelsize': 7, 'legend.fontsize': 7}

with plt.rc_context(style):
    x, y = kappas('baseline_16ch'), kappas('cdan_16ch')
    fig, ax = plt.subplots(figsize=(7.5 * CM, 7.5 * CM))
    ax.scatter(x, y, s=10, alpha=0.7, color='steelblue', edgecolor='none')
    ax.plot([-0.2, 1.05], [-0.2, 1.05], 'k--', lw=0.8, label='equal kappa')
    ax.set_xlabel('Kappa, no-adaptation baseline (16 ch)')
    ax.set_ylabel('Kappa, CDAN (16 ch)')
    ax.set_xlim(-0.2, 1.05)
    ax.set_ylim(-0.2, 1.05)
    ax.set_aspect('equal')
    ax.legend(loc='upper left', frameon=False)
    fig.tight_layout()
    for ext, kw in (('png', {'dpi': 1200}), ('svg', {}), ('pdf', {})):
        fig.savefig(figures_path / f'v2_cdan16_vs_baseline16.{ext}', bbox_inches='tight', **kw)
    plt.show()